[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_11_12/10_tag_11_12_kausale_dilatierte_faltungen_wavenet.ipynb)

# Kausale und dilatierte Faltungen in WaveNet

## Lernziele

WaveNet verarbeitet zeitliche Abhängigkeiten mit Faltungen und benötigt dafür keinen rekurrenten Hidden State. In diesem Notebook entsteht aus genau 5.000 synthetischen Temperaturwerten eine Ein-Schritt-Regressionsaufgabe: Aus 64 vergangenen Werten wird der unmittelbar folgende Wert geschätzt. Danach wird dieselbe Vorhersage wiederholt autoregressiv verwendet.

Am Ende kannst du erklären, was eine kausale eindimensionale Faltung ist, weshalb sie keine Zukunft lesen darf, was die Dilationsrate verändert und wie sich das Receptive Field aus Kernelgröße und Dilationsraten ergibt. An konkreten Indizes vergleichst du vier nicht dilatierte Schichten mit einem Stapel der Dilationsraten 1, 2, 4 und 8. Zwei fair initialisierte Modelle zeigen anschließend, wann die größere historische Reichweite nützlich sein kann und weshalb Fehler bei einer Mehrschrittprognose trotzdem anwachsen können.

## Bewusste Abgrenzung zur vollständigen WaveNet-Architektur

Dieses Notebook implementiert **nicht** die vollständige ursprüngliche WaveNet-Architektur für Audiosignale. Es übernimmt nur die zentralen Prinzipien: kausale eindimensionale Faltungen, dilatierte und stapelbare Faltungsschichten sowie autoregressive Vorhersage. Das Beispiel bleibt bewusst klein, damit jede Tensorform und jeder erreichbare Zeitindex nachvollziehbar ist.

Im Hauptbeispiel fehlen daher Gated Activation Units, Skip Connections, umfangreiche Residual Blocks, quantisierte Audioausgaben und Mixture-of-Logistics-Ausgaben. Diese Bausteine sind für leistungsfähige Audio-Modelle wichtig, würden hier aber die Kernfrage verdecken: Wie erreicht ein Faltungsnetz vergangene Werte, ohne einen rekurrenten Zustand Schritt für Schritt weiterzureichen?

## Technische Vorbereitung und Reproduzierbarkeit

NumPy erzeugt und verarbeitet die Zeitreihe, Pandas stellt Ergebnistabellen dar und Matplotlib zeichnet alle Diagramme. TensorFlow/Keras baut die beiden Conv1D-Modelle. Feste Seeds und deterministische TensorFlow-Operationen machen den Lauf möglichst reproduzierbar. Geringe numerische Unterschiede zwischen Hardwareplattformen können dennoch auftreten.

Alle Diagramme verwenden ein gemeinsames, gut lesbares Layout. Als Temperatureinheit verwenden wir im synthetischen Beispiel **°C**. Es werden keine Dateien heruntergeladen und keine externen Datenquellen benötigt; das Notebook ist damit auch in Google Colab von oben nach unten ausführbar.

In [ ]:
import os

# Die Umgebungsvariablen werden vor dem TensorFlow-Import gesetzt.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_DETERMINISTIC_OPS"] = "1"

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from IPython.display import Markdown, display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.get_logger().setLevel("ERROR")
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    # Ältere TensorFlow-Versionen kennen diese optionale Einstellung eventuell nicht.
    pass

pd.set_option("display.precision", 4)
plt.rcParams.update({
    "figure.figsize": (10, 4.8),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10.5,
    "figure.dpi": 110,
})

print(f"TensorFlow-Version: {tf.__version__}")
print(f"Fester Seed: {SEED}")

## Aufbau der synthetischen Temperaturreihe

Die beobachtete Reihe besteht aus einem Grundniveau, einer kurzfristigen Sinuskomponente mit Periode 24, einer langsameren Komponente mit Periode 168 und einem latenten Störprozess. Für diesen Prozess gilt näherungsweise

$$e_t = 0{,}40e_{t-1} + 0{,}42e_{t-12} + \varepsilon_t.$$

Damit wirken sowohl der unmittelbar vorherige Störwert als auch ein Wert zwölf Schritte zuvor. Die Summe der Beträge beider Koeffizienten liegt unter 1; zusammen mit moderatem gaußschem Rauschen bleibt der Prozess stabil. Die Lag-12-Komponente ist absichtlich relevant: Das spätere Modell A erreicht nur fünf, Modell B dagegen sechzehn ursprüngliche Eingabewerte.

Obwohl es nur ein Merkmal gibt, besitzt die Rohzeitreihe die zweidimensionale Form `(5000, 1)`. Conv1D erwartet neben der Zeitachse ausdrücklich eine Kanal- beziehungsweise Merkmalsdimension; `(5000,)` würde diese Information nicht enthalten.

In [ ]:
ANZAHL_ZEITSCHRITTE = 5_000
zeit = np.arange(ANZAHL_ZEITSCHRITTE)
rng = np.random.default_rng(SEED)

# Moderates Rauschen treibt den stabilen Störprozess an.
innovationen = rng.normal(loc=0.0, scale=0.45, size=ANZAHL_ZEITSCHRITTE)
stoerprozess = np.zeros(ANZAHL_ZEITSCHRITTE, dtype=np.float64)

# Neben Lag 1 wird bewusst Lag 12 eingebaut. Diese Information liegt später
# außerhalb des Receptive Fields von Modell A, aber innerhalb von Modell B.
for t in range(12, ANZAHL_ZEITSCHRITTE):
    stoerprozess[t] = (
        0.40 * stoerprozess[t - 1]
        + 0.42 * stoerprozess[t - 12]
        + innovationen[t]
    )

grundniveau = 15.0
kurzfristig = 2.8 * np.sin(2 * np.pi * zeit / 24)
langfristig = 1.6 * np.sin(2 * np.pi * zeit / 168)
temperatur = grundniveau + kurzfristig + langfristig + stoerprozess

# Die explizite zweite Dimension ist die eine Merkmals-/Kanaldimension.
zeitreihe = temperatur.astype(np.float32).reshape(-1, 1)
assert zeitreihe.shape == (5000, 1)
assert np.all(np.isfinite(zeitreihe))

print("Form der ursprünglichen Zeitreihe:", zeitreihe.shape)
print(f"Wertebereich: {zeitreihe.min():.2f} °C bis {zeitreihe.max():.2f} °C")

plt.figure(figsize=(12, 4.5))
plt.plot(zeit[:500], zeitreihe[:500, 0], color="tab:blue", linewidth=1.4,
         label="synthetische Temperatur")
plt.title("Ausschnitt der synthetischen Temperaturreihe (erste 500 Schritte)")
plt.xlabel("Zeitindex")
plt.ylabel("Temperatur (°C)")
plt.legend()
plt.tight_layout()
plt.show()

## Chronologische Aufteilung und Standardisierung ohne Data Leakage

Die ersten 3.500 Rohwerte bilden das Training, die folgenden 750 die Validierung und die letzten 750 den Test. Eine zufällige Mischung über alle Zeitpunkte wäre unrealistisch: Dann könnten spätere Beobachtungen im Training landen, während frühere Beobachtungen getestet werden. Insbesondere überlappende zeitliche Muster würden Informationen aus der Zukunft in die Modellentwicklung tragen und die Bewertung zu optimistisch machen.

Mittelwert und Standardabweichung werden deshalb **ausschließlich aus dem Trainingsabschnitt** berechnet. Genau dieselbe Transformation wird danach auf Training, Validierung und Test angewendet. Eigene Validierungs- oder Teststatistiken würden Informationen aus diesen zurückgehaltenen Abschnitten in die Vorverarbeitung einschleusen. Für verständliche Metriken und Diagramme werden Modellwerte später mit gespeichertem Trainingsmittelwert und gespeicherter Trainingsstandardabweichung wieder in °C zurücktransformiert.

## Fenster, Zielwerte und Tensorformen

Jedes Fenster enthält genau 64 vergangene Werte und sein Ziel ist der direkt folgende Wert. Das ist eine **Sequence-to-Vector-Regression**: Eine Eingabe hat die Form `(64, 1)`, das Ziel die Form `(1,)`. Fenster werden getrennt innerhalb jedes Datenabschnitts erzeugt; dadurch überschreitet kein Fenster eine Split-Grenze.

Ein Trainings-Batch der Größe 32 besitzt die Form `(32, 64, 1)`: 32 Sequenzen, jeweils 64 Zeitschritte und pro Schritt ein Temperaturmerkmal. Allgemein lautet die Conv1D-Eingabeform `(Batchgröße, Sequenzlänge, Anzahl Merkmale)`. Die folgende Zelle prüft die exakten geforderten Datensatzformen, zeigt die Formen einer Sequenz, eines Zeitschritts und eines Zielwerts und visualisiert ein konkretes standardisiertes Trainingsfenster samt Ziel.

In [ ]:
# Strikte chronologische Aufteilung der Rohwerte, ohne zufälliges Mischen.
train_roh = zeitreihe[:3500]
val_roh = zeitreihe[3500:4250]
test_roh = zeitreihe[4250:]

assert train_roh.shape == (3500, 1)
assert val_roh.shape == (750, 1)
assert test_roh.shape == (750, 1)

# Nur das Training bestimmt Lage und Skala aller drei Abschnitte.
train_mittelwert = train_roh.mean(axis=0)
train_standardabweichung = train_roh.std(axis=0)
assert train_mittelwert.shape == train_standardabweichung.shape == (1,)
assert np.all(train_standardabweichung > 0)

train_skaliert = (train_roh - train_mittelwert) / train_standardabweichung
val_skaliert = (val_roh - train_mittelwert) / train_standardabweichung
test_skaliert = (test_roh - train_mittelwert) / train_standardabweichung


def erzeuge_fenster(abschnitt, fensterlaenge=64):
    """Erzeugt innerhalb eines Abschnitts Sequenzen und explizit zweidimensionale Ziele."""
    X = np.stack([
        abschnitt[start:start + fensterlaenge]
        for start in range(len(abschnitt) - fensterlaenge)
    ])
    y = np.stack([
        abschnitt[start + fensterlaenge]
        for start in range(len(abschnitt) - fensterlaenge)
    ])
    return X.astype(np.float32), y.astype(np.float32)


FENSTERLAENGE = 64
X_train, y_train = erzeuge_fenster(train_skaliert, FENSTERLAENGE)
X_val, y_val = erzeuge_fenster(val_skaliert, FENSTERLAENGE)
X_test, y_test = erzeuge_fenster(test_skaliert, FENSTERLAENGE)

# Die Assertions dokumentieren die geforderte Passung von Zeit-, Kanal- und Zielachse.
assert X_train.shape == (3436, 64, 1) and y_train.shape == (3436, 1)
assert X_val.shape == (686, 64, 1) and y_val.shape == (686, 1)
assert X_test.shape == (686, 64, 1) and y_test.shape == (686, 1)

print("Rohdatenformen:")
print("  Training   ", train_roh.shape)
print("  Validierung", val_roh.shape)
print("  Test       ", test_roh.shape)
print(f"Trainingsmittelwert: {train_mittelwert[0]:.4f} °C")
print(f"Trainingsstandardabweichung: {train_standardabweichung[0]:.4f} °C")
print("\nFensterformen:")
print("X_train.shape      =", X_train.shape)
print("y_train.shape      =", y_train.shape)
print("X_val.shape        =", X_val.shape)
print("y_val.shape        =", y_val.shape)
print("X_test.shape       =", X_test.shape)
print("y_test.shape       =", y_test.shape)
print("X_train[0].shape   =", X_train[0].shape)
print("X_train[0, 0].shape=", X_train[0, 0].shape)
print("y_train[0].shape   =", y_train[0].shape)
print("\nAusschnitt X_train[0, :6, 0] =", np.round(X_train[0, :6, 0], 3))
print("Zugehöriger Zielwert y_train[0] =", np.round(y_train[0], 3))

plt.figure(figsize=(11, 4.5))
plt.plot(np.arange(64), X_train[0, :, 0], marker="o", markersize=3,
         linewidth=1.3, label="64 Eingabewerte")
plt.scatter(64, y_train[0, 0], color="tab:red", s=65, zorder=4,
            label="Zielwert (nächster Schritt)")
plt.title("Beispiel eines standardisierten Trainingsfensters mit Zielwert")
plt.xlabel("Position relativ zum Fensterstart")
plt.ylabel("Standardisierte Temperatur")
plt.legend()
plt.tight_layout()
plt.show()

## Kausale Faltung: keine Information aus der Zukunft

Eine gewöhnliche eindimensionale Faltung kann abhängig vom Padding Werte links und rechts einer Position verwenden. Bei einer Prognose darf die Ausgabe $y_t$ jedoch ausschließlich von $x_t, x_{t-1}, x_{t-2},\ldots$ abhängen. $x_{t+1}, x_{t+2},\ldots$ wären Zukunftsinformationen und sind verboten. Keras setzt diese Richtung mit `padding="causal"` um: Fehlende Werte werden nur links am Sequenzanfang aufgefüllt, die zeitliche Länge bleibt erhalten.

Das transparente Beispiel verwendet $x=[1,2,\ldots,32]$ in der Form `(32, 1)` und betrachtet den letzten Index $t=31$. Rechts davon existieren im Array keine Werte; hypothetische Positionen $x_{32}$ und später wären grundsätzlich unzulässig. Die vier Teilplots markieren für Kernelgröße 2 jeweils genau die beiden Positionen, die eine einzelne Schicht direkt liest. Die weiteren Diagramme zeigen anschließend den durch mehrere Schichten indirekt erreichbaren Bereich.

## Dilationsrate und Receptive Field

Bei Kernelgröße 2 liest eine Schicht mit Dilationsrate 1 die Positionen $x_t$ und $x_{t-1}$. Mit Rate 2 sind es $x_t$ und $x_{t-2}$, mit Rate 4 entsprechend $x_t$ und $x_{t-4}$ und mit Rate 8 schließlich $x_t$ und $x_{t-8}$. Die Kernelgröße und damit die Zahl der Filtergewichte bleibt gleich; die Dilationsrate beschreibt nur den Abstand der gelesenen Positionen.

Das **Receptive Field** ist die Zahl ursprünglicher Eingabezeitschritte, die eine Ausgabe maximal beeinflussen können. Für den hier verwendeten seriellen Stapel mit Stride 1, ohne Pooling, Kernelgröße $k$ und Dilationsraten $d_1,\ldots,d_L$ gilt

$$R = 1 + (k-1)\sum_{l=1}^{L} d_l.$$

Mit $k=2$ und `[1, 2, 4, 8]` ergibt sich $R=1+(2-1)(1+2+4+8)=16$. Für $t=31$ sind dadurch alle Eingaben $x_{16}$ bis $x_{31}$ erreichbar. Vier Raten `[1, 1, 1, 1]` erreichen dagegen nur $R=5$, also $x_{27}$ bis $x_{31}$. Ein größeres Receptive Field ist kein Qualitätsversprechen; es hilft nur, wenn weiter zurückliegende Werte tatsächlich relevante Information tragen.

In [ ]:
def receptive_field(kernelgroesse, dilationsraten):
    """Berechnet das Receptive Field des seriellen, kausalen Schichtstapels."""
    return 1 + (kernelgroesse - 1) * sum(dilationsraten)


def erreichbare_eingaben(ausgabeindex, kernelgroesse, dilationsraten):
    """Verfolgt alle Abhängigkeitspfade rückwärts bis zur ursprünglichen Eingabe."""
    erreichbar = {ausgabeindex}
    for dilation in reversed(dilationsraten):
        erreichbar = {
            position - offset * dilation
            for position in erreichbar
            for offset in range(kernelgroesse)
        }
    return sorted(erreichbar)


KERNELGROESSE = 2
DILATIONEN_A = [1, 1, 1, 1]
DILATIONEN_B = [1, 2, 4, 8]
RF_A = receptive_field(KERNELGROESSE, DILATIONEN_A)
RF_B = receptive_field(KERNELGROESSE, DILATIONEN_B)

# Das manuelle Beispiel behält die explizite Kanaldimension bei.
x_manuell = np.arange(1, 33, dtype=np.float32).reshape(32, 1)
assert x_manuell.shape == (32, 1)
assert RF_A == 5 and RF_B == 16
assert erreichbare_eingaben(31, 2, DILATIONEN_A) == list(range(27, 32))
assert erreichbare_eingaben(31, 2, DILATIONEN_B) == list(range(16, 32))

print("Manuelle Sequenz x =", x_manuell[:, 0].astype(int).tolist())
print("Form der manuellen Sequenz:", x_manuell.shape)

varianten = []
for anzahl in range(1, 6):
    verdoppelt = [2**i for i in range(anzahl)]
    ohne_dilatation = [1] * anzahl
    varianten.extend([
        {"Variante": "verdoppelte Dilatation", "Dilationsraten": str(verdoppelt),
         "Schichten": anzahl, "Receptive Field": receptive_field(2, verdoppelt)},
        {"Variante": "alle Raten = 1", "Dilationsraten": str(ohne_dilatation),
         "Schichten": anzahl, "Receptive Field": receptive_field(2, ohne_dilatation)},
    ])

rf_tabelle = pd.DataFrame(varianten)
display(rf_tabelle)
print("\nKonkrete Rechnung für Modell B:")
print("R = 1 + (2 - 1) · (1 + 2 + 4 + 8) =", RF_B)
print("Erreichbar bei t=31, Modell B:", erreichbare_eingaben(31, 2, DILATIONEN_B))
print("Erreichbar bei t=31, Modell A:", erreichbare_eingaben(31, 2, DILATIONEN_A))

# Plot 1: Direkte Lesezugriffe jeweils einer einzelnen dilatierten Schicht.
fig, achsen = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
for achse, dilation in zip(achsen, [1, 2, 4, 8]):
    vorhandene = np.arange(32)
    direkt = [31 - dilation, 31]
    achse.scatter(vorhandene, np.zeros_like(vorhandene), color="lightgray", s=34,
                  label="vorhandene Eingaben $x_0$ bis $x_{31}$")
    achse.scatter(direkt, [0, 0], color="tab:red", s=75, zorder=3,
                  label=f"direkt gelesen: $x_{{{direkt[0]}}}$, $x_{{31}}$")
    achse.scatter(np.arange(32, 36), np.zeros(4), marker="x", color="black", s=52,
                  label="hypothetische Zukunft: verboten und nicht vorhanden")
    achse.set_title(f"Einzelne kausale Faltung: Dilationsrate {dilation}")
    achse.set_ylabel("Eingabe")
    achse.set_yticks([])
    achse.legend(loc="upper left", fontsize=8, ncol=2)
achsen[-1].set_xlabel("Zeitindex (betrachtete Ausgabe bei t = 31)")
achsen[-1].set_xticks(np.arange(36))
achsen[-1].set_xticklabels([f"$x_{{{i}}}$" for i in range(36)], rotation=90)
fig.suptitle("Direkte Zugriffe bei Kernelgröße 2 – Zukunftswerte werden nie gelesen", y=1.01)
fig.tight_layout()
plt.show()


def plot_erreichbarer_bereich(erreichbar, titel, farbe):
    """Markiert den gesamten historischen Bereich eines Schichtstapels."""
    plt.figure(figsize=(12, 2.8))
    indizes = np.arange(32)
    plt.scatter(indizes, np.zeros(32), color="lightgray", s=45, label="nicht erreichbar")
    plt.scatter(erreichbar, np.zeros(len(erreichbar)), color=farbe, s=78,
                label="durch den Stapel erreichbar")
    plt.scatter([31], [0], marker="*", color="gold", edgecolor="black", s=190,
                zorder=4, label="betrachtete Ausgabe t = 31")
    plt.title(titel)
    plt.xlabel("Ursprünglicher Eingabeindex")
    plt.ylabel("Eingabeebene")
    plt.yticks([])
    plt.xticks(indizes, [f"$x_{{{i}}}$" for i in indizes], rotation=90)
    plt.legend(loc="upper left", ncol=3, fontsize=8)
    plt.tight_layout()
    plt.show()


plot_erreichbarer_bereich(
    erreichbare_eingaben(31, 2, DILATIONEN_B),
    "Gesamtes Receptive Field: Dilationsraten [1, 2, 4, 8] erreichen x₁₆ bis x₃₁",
    "tab:blue",
)
plot_erreichbarer_bereich(
    erreichbare_eingaben(31, 2, DILATIONEN_A),
    "Nicht dilatierter Stapel: vier Raten 1 erreichen nur x₂₇ bis x₃₁",
    "tab:orange",
)

# Plot 4: Verdoppelte Raten erweitern die Reichweite wesentlich schneller.
schichten = np.arange(1, 6)
rf_ohne = [receptive_field(2, [1] * n) for n in schichten]
rf_verdoppelt = [receptive_field(2, [2**i for i in range(n)]) for n in schichten]
plt.figure(figsize=(8.5, 4.8))
plt.plot(schichten, rf_ohne, marker="o", linewidth=2,
         label="alle Dilationsraten = 1")
plt.plot(schichten, rf_verdoppelt, marker="o", linewidth=2,
         label="Dilationsraten 1, 2, 4, 8, 16")
plt.title("Wachstum des Receptive Fields bei Kernelgröße 2")
plt.xlabel("Anzahl Faltungsschichten")
plt.ylabel("Receptive Field (Zeitschritte)")
plt.xticks(schichten)
plt.legend()
plt.tight_layout()
plt.show()

## Abhängigkeitsdiagramm des dilatierten Stapels

Das folgende Diagramm ist keine dekorative Netzskizze, sondern verfolgt die tatsächlich benötigten Positionen rückwärts. Die Conv-Schicht mit Rate 8 an Position 31 liest aus der vorherigen Ebene die Positionen 31 und 23. Von dort werden mit Rate 4, dann Rate 2 und schließlich Rate 1 weitere Pfade aufgefächert. Auf der Eingabeebene entsteht so der lückenlose Bereich 16 bis 31.

Jede horizontale Zeile steht für eine Ebene. Eine Kante verbindet eine Ausgabeposition mit genau den beiden Positionen der vorigen Ebene, die ihr Kernel der Größe 2 liest. Die oberste Zeile zeigt, dass nur die Repräsentation am letzten Zeitpunkt an die finale Ausgabe weitergegeben wird.

In [ ]:
dilationsfolge = [1, 2, 4, 8]
benoetigt = [set() for _ in range(5)]  # Eingabe plus vier Conv-Ebenen
benoetigt[4] = {31}
kanten = []

# Rückwärtsanalyse: Jede Position liest sich selbst und eine um d verschobene Position.
for ebene in range(4, 0, -1):
    dilation = dilationsfolge[ebene - 1]
    for zielposition in sorted(benoetigt[ebene]):
        for quellposition in [zielposition, zielposition - dilation]:
            benoetigt[ebene - 1].add(quellposition)
            kanten.append((ebene - 1, quellposition, ebene, zielposition))

assert sorted(benoetigt[0]) == list(range(16, 32))

ebenen_namen = [
    "Eingabe",
    "kausale Conv d=1",
    "kausale Conv d=2",
    "kausale Conv d=4",
    "kausale Conv d=8",
    "finale Ausgabe",
]
farben = ["#4C78A8", "#72B7B2", "#54A24B", "#ECA82C", "#E45756"]

fig, achse = plt.subplots(figsize=(13, 7))
for unten, x_unten, oben, x_oben in kanten:
    achse.plot([x_unten, x_oben], [unten, oben], color="gray", alpha=0.55, linewidth=1)

for ebene, positionen in enumerate(benoetigt):
    xs = sorted(positionen)
    achse.scatter(xs, [ebene] * len(xs), s=72, color=farben[ebene],
                  edgecolor="white", zorder=3)
    for x_pos in xs:
        praefix = "x" if ebene == 0 else "h"
        achse.text(x_pos, ebene + 0.12, f"{praefix}$_{{{x_pos}}}$", ha="center", fontsize=7)

# Die Modell-Ausgabe wählt nur die letzte Repräsentation der vierten Conv-Schicht.
achse.plot([31, 31], [4, 5], color="gray", linewidth=1.5)
achse.scatter([31], [5], marker="*", s=230, color="gold", edgecolor="black",
              zorder=4, label="ausgewählte finale Ausgabe bei t=31")
achse.set_title("Korrekte zeitliche Abhängigkeit des dilatierten Faltungsstapels")
achse.set_xlabel("Zeitindex")
achse.set_ylabel("Netzebene")
achse.set_xticks(np.arange(16, 32))
achse.set_yticks(np.arange(6), ebenen_namen)
achse.set_xlim(15.4, 31.6)
achse.set_ylim(-0.35, 5.45)
achse.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
fig.tight_layout()
plt.show()

## Zwei Modelle, passende Tensorformen und faire Bedingungen

Beide Modelle erhalten `(64, 1)`. Nach jeder kausalen Conv1D-Schicht mit 16 Filtern lautet die Form `(64, 16)`, weil `padding="causal"` die zeitliche Länge erhält und die Filter 16 neue Kanäle erzeugen. Nach der vierten Faltung wird nur der letzte Zeitschritt gewählt: `(16,)`. `Dense(1)` mit linearer Aktivierung liefert schließlich `(1,)`, den nächsten kontinuierlichen Temperaturwert. Softmax, Sigmoid und Accuracy wären für diese Regression unpassend.

Modell A verwendet viermal Dilationsrate 1 und besitzt $R=5$. Modell B verwendet 1, 2, 4 und 8 und besitzt $R=16$. Alle übrigen Architektur- und Trainingsparameter sind gleich. Ein Basismodell liefert identische Startgewichtstensoren für beide Varianten; Assertions prüfen Form und Werte. Beide Modelle sehen dieselben Fenster in derselben Reihenfolge (`shuffle=False`), nutzen Batchgröße 32, 25 Epochen, Adam mit Lernrate 0,001 und MSE.

MSE eignet sich als Loss, weil große quadratische Fehler beim Optimieren stärker zählen. MAE beschreibt den mittleren absoluten Fehler direkt in °C; RMSE liegt ebenfalls in °C, reagiert aber stärker auf große Abweichungen. Trainingsmetriken werden auf standardisierten Werten berechnet, die abschließenden Test-MAE und Test-RMSE dagegen nach Rücktransformation in die ursprüngliche Temperatureinheit.

In [ ]:
def baue_modell(dilationsraten, name):
    """Erstellt den gemeinsamen kausalen Conv1D-Aufbau mit wählbaren Dilationsraten."""
    eingabe = keras.Input(shape=(64, 1), name="temperaturfenster")
    x = eingabe
    for nummer, dilation in enumerate(dilationsraten, start=1):
        x = layers.Conv1D(
            filters=16,
            kernel_size=2,
            dilation_rate=dilation,
            padding="causal",
            activation="relu",
            name=f"conv_{nummer}_d{dilation}",
        )(x)
    # Für Sequence-to-Vector wird nur die Repräsentation des letzten Zeitpunkts genutzt.
    x = layers.Lambda(lambda tensor: tensor[:, -1, :], name="letzter_zeitpunkt")(x)
    ausgabe = layers.Dense(1, activation="linear", name="temperatur_naechster_schritt")(x)
    return keras.Model(eingabe, ausgabe, name=name)


def kompiliere_modell(modell):
    modell.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="mse",
        metrics=[
            keras.metrics.MeanAbsoluteError(name="mae"),
            keras.metrics.RootMeanSquaredError(name="rmse"),
        ],
    )


def ruecktransformieren(werte):
    """Bringt standardisierte Vorhersagen mit Trainingsstatistiken zurück nach °C."""
    return np.asarray(werte) * train_standardabweichung + train_mittelwert


def prognose_und_metriken(modell, X, y):
    prognose_skaliert = modell.predict(X, verbose=0)
    prognose_c = ruecktransformieren(prognose_skaliert).reshape(-1)
    wahr_c = ruecktransformieren(y).reshape(-1)
    fehler = prognose_c - wahr_c
    return {
        "prognose": prognose_c,
        "wahr": wahr_c,
        "residuen": fehler,
        "mae": float(np.mean(np.abs(fehler))),
        "rmse": float(np.sqrt(np.mean(fehler**2))),
    }


def plot_modellauswertung(history, ergebnis, modellname, farbe):
    """Erzeugt drei bewusst getrennte Diagramme für Loss, Prognose und Residuen."""
    epochen_achse = np.arange(1, len(history.history["loss"]) + 1)
    plt.figure(figsize=(8.5, 4.5))
    plt.plot(epochen_achse, history.history["loss"], label="Training-MSE")
    plt.plot(epochen_achse, history.history["val_loss"], label="Validierungs-MSE")
    plt.title(f"{modellname}: Trainings- und Validierungs-Loss")
    plt.xlabel("Epoche")
    plt.ylabel("MSE (standardisierte Skala)")
    plt.xticks(np.arange(1, len(epochen_achse) + 1, 2))
    plt.legend()
    plt.tight_layout()
    plt.show()

    ausschnitt = 180
    x_achse = np.arange(ausschnitt)
    plt.figure(figsize=(11, 4.5))
    plt.plot(x_achse, ergebnis["wahr"][:ausschnitt], linewidth=1.7,
             label="tatsächliche Temperatur")
    plt.plot(x_achse, ergebnis["prognose"][:ausschnitt], color=farbe,
             linewidth=1.4, label="Ein-Schritt-Prognose")
    plt.title(f"{modellname}: Vorhersage auf einem zusammenhängenden Testabschnitt")
    plt.xlabel("Position im Testabschnitt")
    plt.ylabel("Temperatur (°C)")
    plt.ylim(EIN_SCHRITT_YLIM)
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(11, 3.8))
    plt.axhline(0, color="black", linewidth=1)
    plt.plot(x_achse, ergebnis["residuen"][:ausschnitt], color=farbe,
             linewidth=1.2, label="Vorhersage − tatsächlicher Wert")
    plt.title(f"{modellname}: Residuen auf demselben Testabschnitt")
    plt.xlabel("Position im Testabschnitt")
    plt.ylabel("Residuum (°C)")
    plt.ylim(RESIDUEN_YLIM)
    plt.legend()
    plt.tight_layout()
    plt.show()


# Ein Basismodell erzeugt genau einen Satz Startgewichte für beide Architekturen.
basismodell = baue_modell(DILATIONEN_A, "Basismodell")
startgewichte = [gewicht.copy() for gewicht in basismodell.get_weights()]
modell_a = baue_modell(DILATIONEN_A, "Modell_A_ohne_Dilatation")
modell_b = baue_modell(DILATIONEN_B, "Modell_B_mit_Dilatation")
modell_a.set_weights(startgewichte)
modell_b.set_weights(startgewichte)

assert all(np.allclose(a, b) for a, b in zip(modell_a.get_weights(), modell_b.get_weights()))
assert modell_a.input_shape == modell_b.input_shape == (None, 64, 1)
assert modell_a.output_shape == modell_b.output_shape == (None, 1)
assert modell_a.count_params() == modell_b.count_params()
assert RF_A == 5 and RF_B == 16 and RF_A < 12 <= RF_B

for modell, erwartete_raten in [(modell_a, DILATIONEN_A), (modell_b, DILATIONEN_B)]:
    conv_schichten = [s for s in modell.layers if isinstance(s, layers.Conv1D)]
    assert len(conv_schichten) == 4
    assert all(s.padding == "causal" for s in conv_schichten)
    assert [s.dilation_rate[0] for s in conv_schichten] == erwartete_raten
    assert conv_schichten[0].input.shape[1:] == (64, 1)
    assert conv_schichten[0].output.shape[1:] == (64, 16)
    assert modell.get_layer("letzter_zeitpunkt").output.shape[1:] == (16,)
    assert modell.layers[-1].units == 1
    assert modell.layers[-1].activation.__name__ == "linear"

kompiliere_modell(modell_a)
kompiliere_modell(modell_b)

BATCHGROESSE = 32
EPOCHEN = 25
EIN_SCHRITT_YLIM = (
    float(ruecktransformieren(y_test).min() - 0.8),
    float(ruecktransformieren(y_test).max() + 0.8),
)
residuen_grenze = max(3.0, float(train_standardabweichung[0] * 1.25))
RESIDUEN_YLIM = (-residuen_grenze, residuen_grenze)

print("Identische Startgewichte:", all(
    np.allclose(a, b) for a, b in zip(modell_a.get_weights(), modell_b.get_weights())
))
print("Trainierbare Parameter je Modell:", modell_a.count_params())
print("Modell A: Dilationsraten", DILATIONEN_A, "| Receptive Field", RF_A)
print("Modell B: Dilationsraten", DILATIONEN_B, "| Receptive Field", RF_B)
modell_a.summary()

## Auswertung Modell A – kausal ohne Dilatation

Modell A besitzt vier kausale Conv1D-Schichten mit Dilationsrate 1. Seine finale Repräsentation erreicht bei Kernelgröße 2 nur die letzten fünf Werte des 64er-Fensters. Eine direkte Lag-12-Abhängigkeit liegt außerhalb dieses Bereichs; sie kann nicht über die vier Faltungsschichten bis zur ausgewählten letzten Ausgabe gelangen.

Die nächste Zelle trainiert Modell A ohne Shuffling und wertet es vollständig separat aus. Der Loss-Plot zeigt nur Epochen auf der x-Achse. Prognose und Residuen beziehen sich auf denselben zusammenhängenden Testausschnitt und werden in °C dargestellt. Die anschließend ausgegebene Interpretation nennt die tatsächlich gemessenen Loss-, MAE- und RMSE-Werte dieses Laufs.

In [ ]:
# Modell A sieht alle Batches in chronologischer, deterministischer Reihenfolge.
history_a = modell_a.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHEN,
    batch_size=BATCHGROESSE,
    shuffle=False,
    verbose=0,
)
ergebnis_a = prognose_und_metriken(modell_a, X_test, y_test)

print(f"Finaler Trainings-Loss Modell A: {history_a.history['loss'][-1]:.6f}")
print(f"Finaler Validierungs-Loss Modell A: {history_a.history['val_loss'][-1]:.6f}")
print(f"Test-MAE Modell A: {ergebnis_a['mae']:.4f} °C")
print(f"Test-RMSE Modell A: {ergebnis_a['rmse']:.4f} °C")
plot_modellauswertung(history_a, ergebnis_a, "Modell A – kausal ohne Dilatation", "tab:orange")

minimum_epoche_a = int(np.argmin(history_a.history["val_loss"]) + 1)
display(Markdown(f"""
**Ergebnisinterpretation Modell A.** Das Receptive Field beträgt 5; direkt erreichbar sind
für die nächste Vorhersage nur die fünf jüngsten Positionen des Fensters. Der kleinste
Validierungs-Loss trat in Epoche {minimum_epoche_a} auf, der finale Trainings-/Validierungs-MSE
liegt bei {history_a.history['loss'][-1]:.5f}/{history_a.history['val_loss'][-1]:.5f}.
Auf dem zurücktransformierten Testdatensatz betragen MAE **{ergebnis_a['mae']:.3f} °C** und
RMSE **{ergebnis_a['rmse']:.3f} °C**. Die Residuen um die Nulllinie zeigen die verbleibenden
Ein-Schritt-Fehler; die Lag-12-Komponente kann dieses Modell nicht direkt aus seinem
Receptive Field beziehen.
"""))

## Interpretation von Modell A

Der Abstand zwischen Trainings- und Validierungskurve hilft, Generalisierung und mögliches Overfitting einzuordnen; entscheidend sind nicht einzelne Zacken, sondern Niveau und Verlauf. Der Prognoseplot zeigt, ob kurzfristige Richtung und Amplitude getroffen werden. Im Residuenplot bedeutet die Nulllinie perfekte Vorhersage, positive Werte sind Überschätzungen und negative Werte Unterschätzungen.

Die unmittelbar darüber gerenderte Ergebnisinterpretation verwendet die tatsächlich ausgeführten Werte. Ihre zentrale strukturelle Grenze bleibt unabhängig vom konkreten Loss: Von 64 angebotenen Eingabeschritten können nur die letzten fünf die finale Ausgabe beeinflussen. Damit fehlen Modell A direkte Pfade zur bewusst eingebauten Lag-12-Information.

## Auswertung Modell B – kausal mit Dilatation

Modell B unterscheidet sich nur durch die Dilationsraten 1, 2, 4 und 8. Trotz derselben vier Conv1D-Schichten, Filterzahl, Kernelgröße und Parameterzahl erweitert sich sein Receptive Field auf 16. Die für die Datenerzeugung wichtige Lag-12-Abhängigkeit liegt damit innerhalb des erreichbaren Bereichs.

Training, Batchreihenfolge, Optimierer, Lernrate, Loss, Metriken und Anzahl Epochen bleiben identisch zu Modell A. Die folgenden drei Diagramme sind wieder getrennt und verwenden für Prognose beziehungsweise Residuen dieselben Achsengrenzen wie Modell A. Erst danach werden beide Modelle kompakt verglichen.

In [ ]:
# Modell B startet mit denselben Gewichten und erhält dieselben Batches wie Modell A.
history_b = modell_b.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHEN,
    batch_size=BATCHGROESSE,
    shuffle=False,
    verbose=0,
)
ergebnis_b = prognose_und_metriken(modell_b, X_test, y_test)

print(f"Finaler Trainings-Loss Modell B: {history_b.history['loss'][-1]:.6f}")
print(f"Finaler Validierungs-Loss Modell B: {history_b.history['val_loss'][-1]:.6f}")
print(f"Test-MAE Modell B: {ergebnis_b['mae']:.4f} °C")
print(f"Test-RMSE Modell B: {ergebnis_b['rmse']:.4f} °C")
plot_modellauswertung(history_b, ergebnis_b, "Modell B – kausal mit Dilatation", "tab:blue")

minimum_epoche_b = int(np.argmin(history_b.history["val_loss"]) + 1)
vergleichssatz = (
    "Die größere Reichweite ist in diesem Lauf auch numerisch erkennbar: Modell B hat "
    "den niedrigeren Test-MAE."
    if ergebnis_b["mae"] < ergebnis_a["mae"]
    else "In diesem Lauf führt die größere Reichweite nicht zu einem niedrigeren Test-MAE; "
         "ein größeres Receptive Field garantiert keine bessere Prognose."
)
display(Markdown(f"""
**Ergebnisinterpretation Modell B.** Das Receptive Field beträgt 16 und enthält damit Lag 12.
Der kleinste Validierungs-Loss trat in Epoche {minimum_epoche_b} auf; der finale
Trainings-/Validierungs-MSE liegt bei {history_b.history['loss'][-1]:.5f}/
{history_b.history['val_loss'][-1]:.5f}. Nach Rücktransformation ergeben sich ein Test-MAE
von **{ergebnis_b['mae']:.3f} °C** und ein Test-RMSE von **{ergebnis_b['rmse']:.3f} °C**.
{vergleichssatz} Die Loss- und Residuenplots zeigen zugleich, dass auch das dilatierte
Modell keine fehlerfreie Vorhersage liefert.
"""))

## Interpretation von Modell B

Modell B kann über seine Abhängigkeitspfade auf die letzten 16 Werte zugreifen und damit die gezielt erzeugte Lag-12-Struktur direkt nutzen. Ob dieser theoretische Vorteil praktisch sichtbar wird, entscheiden die gemessenen Testfehler und nicht die Architekturformel allein. Die dynamische Interpretation oberhalb formuliert daher abhängig von den tatsächlich berechneten Werten, ob Modell B in diesem Lauf beim Test-MAE vorne liegt.

Auch bei einem niedrigeren Fehler bleiben Residuen bestehen: Rauschen ist nicht perfekt vorhersagbar, eine kleine Architektur bildet nicht jede Beziehung exakt ab und ein größeres Receptive Field kann zusätzliche, aber nicht automatisch nützliche Werte einschließen. Genau deshalb werden Reichweite und Vorhersagegüte getrennt beurteilt.

## Direkter Modellvergleich

Erst nach den getrennten Auswertungen fasst die folgende Tabelle Architektur und Ergebnisse zusammen. Trainings- und Validierungs-Loss stehen auf der standardisierten Skala; Test-MAE und Test-RMSE sind zurücktransformiert und daher in °C interpretierbar. Zwei kleine, getrennte Teilplots vermeiden es, MAE und RMSE als identische Größen zu behandeln.

Die Dilationsrate verändert nur die Abstände zwischen gelesenen Positionen. Sie ergänzt keine Kernelgewichte. Weil beide Modelle dieselben Filterzahlen, Kernelgrößen und Schichtformen besitzen, ist ihre trainierbare Parameterzahl identisch, obwohl ihre Receptive Fields 5 und 16 betragen.

In [ ]:
vergleich = pd.DataFrame([
    {
        "Modellname": "Modell A – ohne Dilatation",
        "Dilationsraten": str(DILATIONEN_A),
        "Kernelgröße": KERNELGROESSE,
        "Faltungsschichten": 4,
        "Receptive Field": RF_A,
        "trainierbare Parameter": modell_a.count_params(),
        "finaler Trainings-Loss": history_a.history["loss"][-1],
        "finaler Validierungs-Loss": history_a.history["val_loss"][-1],
        "Test-MAE (°C)": ergebnis_a["mae"],
        "Test-RMSE (°C)": ergebnis_a["rmse"],
    },
    {
        "Modellname": "Modell B – mit Dilatation",
        "Dilationsraten": str(DILATIONEN_B),
        "Kernelgröße": KERNELGROESSE,
        "Faltungsschichten": 4,
        "Receptive Field": RF_B,
        "trainierbare Parameter": modell_b.count_params(),
        "finaler Trainings-Loss": history_b.history["loss"][-1],
        "finaler Validierungs-Loss": history_b.history["val_loss"][-1],
        "Test-MAE (°C)": ergebnis_b["mae"],
        "Test-RMSE (°C)": ergebnis_b["rmse"],
    },
])
display(vergleich.round(4))

assert vergleich.loc[0, "trainierbare Parameter"] == vergleich.loc[1, "trainierbare Parameter"]

fig, achsen = plt.subplots(1, 2, figsize=(9.5, 4.3))
modellkurz = ["A", "B"]
mae_werte = [ergebnis_a["mae"], ergebnis_b["mae"]]
rmse_werte = [ergebnis_a["rmse"], ergebnis_b["rmse"]]

achsen[0].bar(modellkurz, mae_werte, color=["tab:orange", "tab:blue"])
achsen[0].set_title("Vergleich des Test-MAE")
achsen[0].set_xlabel("Modell")
achsen[0].set_ylabel("MAE (°C)")
for i, wert in enumerate(mae_werte):
    achsen[0].text(i, wert, f"{wert:.3f}", ha="center", va="bottom")

achsen[1].bar(modellkurz, rmse_werte, color=["tab:orange", "tab:blue"])
achsen[1].set_title("Vergleich des Test-RMSE")
achsen[1].set_xlabel("Modell")
achsen[1].set_ylabel("RMSE (°C)")
for i, wert in enumerate(rmse_werte):
    achsen[1].text(i, wert, f"{wert:.3f}", ha="center", va="bottom")

fig.suptitle("Kompakter Metrikvergleich auf der ursprünglichen Temperaturskala")
fig.tight_layout()
plt.show()

bestes_modell = "Modell B" if ergebnis_b["mae"] < ergebnis_a["mae"] else "Modell A"
display(Markdown(
    f"**Ausgeführter Vergleich:** {bestes_modell} erzielt in diesem Lauf den niedrigeren "
    f"Test-MAE. Die exakten Werte stehen in der Tabelle; die identische Parameterzahl "
    f"({modell_a.count_params():,}) bestätigt, dass Dilatation die Reichweite und nicht "
    "die Zahl der Kernelgewichte verändert."
))

## Autoregressive Mehrschrittprognose und Fehlerakkumulation

Für die Ein-Schritt-Auswertung erhielt das Modell bei jedem Fenster 64 echte historische Werte. Nun starten beide Modelle mit derselben bekannten 64er-Sequenz aus dem Testabschnitt und prognostizieren 40 Schritte. Nach jedem Schritt wird die Vorhersage rechts angehängt, der älteste Wert entfernt und das neue Fenster erneut eingespeist. Bereits ab Schritt 2 enthält das Fenster also einen selbst erzeugten Wert; später dominieren zunehmend eigene Vorhersagen.

Diese Bedingungen unterscheiden sich vom Ein-Schritt-Test. Kleine Abweichungen verändern zukünftige Eingaben und können sich aufschaukeln. Deshalb garantiert ein guter Ein-Schritt-MAE keine perfekte Langzeitgenerierung. Die Modelle werden zunächst in zwei eigenen Plots mit identischen Achsengrenzen beurteilt. Erst nachdem beide Einzelplots gezeigt wurden, folgt ein kleiner gemeinsamer Vergleich.

In [ ]:
def autoregressive_prognose(modell, startfenster, schritte):
    """Schiebt nach jedem Schritt die eigene Vorhersage in das 64er-Eingabefenster."""
    fenster = startfenster.astype(np.float32).copy()
    vorhersagen = []
    for _ in range(schritte):
        naechster_wert = float(modell(fenster[None, ...], training=False).numpy()[0, 0])
        vorhersagen.append(naechster_wert)
        fenster = np.concatenate([fenster[1:], [[naechster_wert]]], axis=0)
    return np.asarray(vorhersagen, dtype=np.float32)


AR_START = 400
AR_SCHRITTE = 40
startfenster = test_skaliert[AR_START:AR_START + FENSTERLAENGE]
zukunft_wahr_skaliert = test_skaliert[
    AR_START + FENSTERLAENGE:AR_START + FENSTERLAENGE + AR_SCHRITTE
]
assert startfenster.shape == (64, 1)
assert zukunft_wahr_skaliert.shape == (40, 1)

# Beide Prognosen werden jetzt berechnet, damit die getrennten Plots exakt dieselben Grenzen nutzen.
ar_a_skaliert = autoregressive_prognose(modell_a, startfenster, AR_SCHRITTE)
ar_b_skaliert = autoregressive_prognose(modell_b, startfenster, AR_SCHRITTE)
start_c = ruecktransformieren(startfenster).reshape(-1)
zukunft_wahr_c = ruecktransformieren(zukunft_wahr_skaliert).reshape(-1)
ar_a_c = ruecktransformieren(ar_a_skaliert).reshape(-1)
ar_b_c = ruecktransformieren(ar_b_skaliert).reshape(-1)

alle_ar_werte = np.concatenate([start_c, zukunft_wahr_c, ar_a_c, ar_b_c])
ar_rand = 0.08 * (alle_ar_werte.max() - alle_ar_werte.min())
AR_YLIM = (float(alle_ar_werte.min() - ar_rand), float(alle_ar_werte.max() + ar_rand))
ar_mae_a = float(np.mean(np.abs(ar_a_c - zukunft_wahr_c)))

x_start = np.arange(-63, 1)
x_zukunft = np.arange(1, AR_SCHRITTE + 1)
plt.figure(figsize=(11, 4.8))
plt.plot(x_start, start_c, color="gray", linewidth=1.5, label="bekannte Startsequenz")
plt.plot(x_zukunft, zukunft_wahr_c, color="black", linewidth=2,
         label="tatsächliche Zukunft")
plt.plot(x_zukunft, ar_a_c, color="tab:orange", linewidth=1.8,
         label="autoregressiv: Modell A")
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
plt.title("Autoregressive 40-Schritt-Prognose – Modell A")
plt.xlabel("Schritt relativ zum Prognosestart")
plt.ylabel("Temperatur (°C)")
plt.ylim(AR_YLIM)
plt.legend()
plt.tight_layout()
plt.show()

frueh_a = float(np.mean(np.abs(ar_a_c[:10] - zukunft_wahr_c[:10])))
spaet_a = float(np.mean(np.abs(ar_a_c[-10:] - zukunft_wahr_c[-10:])))
entwicklung_a = "nimmt zu" if spaet_a > frueh_a else "nimmt in diesem Ausschnitt nicht zu"
display(Markdown(f"""
**Autoregressive Interpretation Modell A.** Über 40 Schritte beträgt der MAE
**{ar_mae_a:.3f} °C**. Der mittlere absolute Fehler der ersten zehn Schritte ist
{frueh_a:.3f} °C, jener der letzten zehn {spaet_a:.3f} °C; er {entwicklung_a}.
Die gezeichnete Kurve zeigt, welche periodischen Schwankungen erhalten bleiben und wo
das Modell abdriftet. Da sein Receptive Field nur 5 beträgt, steht die Lag-12-Struktur
auch während der Generierung nicht direkt zur Verfügung.
"""))

In [ ]:
ar_mae_b = float(np.mean(np.abs(ar_b_c - zukunft_wahr_c)))

plt.figure(figsize=(11, 4.8))
plt.plot(x_start, start_c, color="gray", linewidth=1.5, label="bekannte Startsequenz")
plt.plot(x_zukunft, zukunft_wahr_c, color="black", linewidth=2,
         label="tatsächliche Zukunft")
plt.plot(x_zukunft, ar_b_c, color="tab:blue", linewidth=1.8,
         label="autoregressiv: Modell B")
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
plt.title("Autoregressive 40-Schritt-Prognose – Modell B")
plt.xlabel("Schritt relativ zum Prognosestart")
plt.ylabel("Temperatur (°C)")
plt.ylim(AR_YLIM)
plt.legend()
plt.tight_layout()
plt.show()

frueh_b = float(np.mean(np.abs(ar_b_c[:10] - zukunft_wahr_c[:10])))
spaet_b = float(np.mean(np.abs(ar_b_c[-10:] - zukunft_wahr_c[-10:])))
entwicklung_b = "nimmt zu" if spaet_b > frueh_b else "nimmt in diesem Ausschnitt nicht zu"
reichweiten_text = (
    "Die größere historische Reichweite geht in diesem Ausschnitt mit dem kleineren "
    "Mehrschritt-MAE einher."
    if ar_mae_b < ar_mae_a
    else "Die größere historische Reichweite führt in diesem Ausschnitt nicht zum "
         "kleineren Mehrschritt-MAE."
)
display(Markdown(f"""
**Autoregressive Interpretation Modell B.** Über 40 Schritte beträgt der MAE
**{ar_mae_b:.3f} °C**. Der Fehler der ersten zehn Schritte liegt bei {frueh_b:.3f} °C,
der letzten zehn bei {spaet_b:.3f} °C; er {entwicklung_b}. {reichweiten_text}
Die eigene Vorhersage wird in jedem Schritt zurückgeführt: Selbst ein besserer
Ein-Schritt-Fehler verhindert daher keine Abweichungen im längeren Verlauf.
"""))

# Erst nach beiden Einzelplots folgt der kompakte gemeinsame Vergleich.
plt.figure(figsize=(10, 4.5))
plt.plot(x_zukunft, zukunft_wahr_c, color="black", linewidth=2,
         label="tatsächliche Zukunft")
plt.plot(x_zukunft, ar_a_c, color="tab:orange", linewidth=1.5, label="Modell A")
plt.plot(x_zukunft, ar_b_c, color="tab:blue", linewidth=1.5, label="Modell B")
plt.title("Kompakter Vergleich der autoregressiven Prognosen")
plt.xlabel("Autoregressiver Vorhersageschritt")
plt.ylabel("Temperatur (°C)")
plt.ylim(AR_YLIM)
plt.legend()
plt.tight_layout()
plt.show()

## Autoregressive Ergebnisse richtig lesen

Die beiden Einzelplots verwenden dieselbe bekannte Startsequenz, dieselbe tatsächliche Zukunft und identische Achsengrenzen. Die direkt unter jedem Plot gerenderte Auswertung nennt getrennt den 40-Schritt-MAE sowie die Fehler in den ersten und letzten zehn Schritten. So wird sichtbar, wie schnell sich die jeweilige Prognose entfernt und ob die periodische Struktur erhalten bleibt. Der gemeinsame Plot dient erst danach als kompakte Gegenüberstellung.

Ein möglicher Vorteil des größeren Receptive Fields darf nicht mit Stabilität über beliebig viele Schritte verwechselt werden. Beim Training wurden echte Fenster verwendet, autoregressiv entstehen dagegen zunehmend Eingabekombinationen aus Modellwerten. Die ausgegebenen Kurven und Metriken dieses Laufs sind deshalb die Grundlage der Interpretation, nicht die Annahme, dass Modell B allein wegen seiner Dilatation immer gewinnen müsse.

## Abschließende Kernaussagen

1. Eine kausale Faltung verwendet keine zukünftigen Eingabewerte: Die Ausgabe bei $t$ hängt nur von $x_t$ und früheren Werten ab. `padding="causal"` ergänzt dafür ausschließlich links Padding.
2. Die Dilationsrate vergrößert die Abstände zwischen den vom Kernel gelesenen Positionen, aber nicht die Anzahl seiner Gewichte. Die identische Parameterzahl in der Vergleichstabelle bestätigt das für beide Modelle.
3. Das Receptive Field beschreibt den historischen Bereich, der eine Ausgabe beeinflussen kann. Die Indexdiagramme zeigen konkret: Vier nicht dilatierte Schichten mit Kernelgröße 2 erreichen fünf Werte ($x_{27}$ bis $x_{31}$), die Raten 1, 2, 4 und 8 dagegen sechzehn ($x_{16}$ bis $x_{31}$).
4. Das dilatierte Modell kann dadurch die relevante Lag-12-Abhängigkeit direkt erreichen; Modell A kann es nicht. Ob das praktisch hilft, zeigen die ausgeführten Loss-Kurven, Test-MAE/RMSE und Residuen. Ein größeres Receptive Field garantiert grundsätzlich keine bessere Vorhersage.
5. WaveNet benötigt keinen rekurrenten Hidden State. Faltungsausgaben verschiedener Zeitpunkte lassen sich im Training besser parallelisieren als zeitlich voneinander abhängige RNN-Zustände.
6. Aus der Ein-Schritt-Prognose wird durch Zurückführen jedes neuen Modellwerts eine autoregressive Mehrschrittprognose. Die getrennten 40-Schritt-Plots und ihre berechneten Fehler zeigen, dass sich kleine Abweichungen ansammeln können; eine gute Ein-Schritt-Prognose garantiert daher keine fehlerfreie langfristige Generierung.